# Vision-IA · CNN propia en Colab

Este notebook está preparado, **sin entrenamiento ejecutado**. Tú ejecutarás las celdas manualmente. No hay pesos preentrenados ni descargas de modelos.

1. En tu Google Drive crea `Vision-IA/` y sube `Vision-IA_Colab_A.zip`.
2. Abre este notebook en Colab y selecciona un entorno con GPU (T4 si está disponible).
3. Ejecuta las celdas en orden. Autoriza el montaje de Drive solo cuando decidas continuar.
4. La celda **7 · Entrenar** inicia el entrenamiento real. Las anteriores solo preparan y verifican.

No se cambian las particiones. Test queda reservado para el evaluador independiente.


## 1 · Comprobar el entorno y la GPU

Ejecuta esta celda. Si no hay GPU, cambia el tipo de entorno antes de entrenar. El notebook no solicita una GPU específica por código: usa la que asigne Colab.


In [ ]:
import sys
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU asignada:', torch.cuda.get_device_name(0))
else:
    print('Selecciona una GPU en la configuración de Colab; no se iniciará entrenamiento en CPU por accidente.')


## 2 · Autorizar el montaje de Drive

Marca `AUTORIZAR_MONTAJE` y ejecuta **solo cuando quieras conceder acceso**. Colab mostrará su diálogo de autorización. No hay rutas personales de Windows.


In [ ]:
AUTORIZAR_MONTAJE = False #@param {type:"boolean"}
if not AUTORIZAR_MONTAJE:
    raise RuntimeError('Marca AUTORIZAR_MONTAJE y vuelve a ejecutar esta celda cuando autorices el acceso.')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')


## 3 · Localizar, verificar y extraer el paquete

El valor por defecto corresponde a la carpeta `Vision-IA/` que creaste en Mi unidad. Si lo subiste directamente a Mi unidad, también se localizará. Si está en otra carpeta, edita la ruta relativa. Se extrae en una carpeta temporal nueva y se comprueban las huellas de todos los archivos.


In [ ]:
RUTA_ZIP_EN_DRIVE = 'Vision-IA/Vision-IA_Colab_A.zip' #@param {type:"string"}
import hashlib
import json
import os
import tempfile
import zipfile
from pathlib import PurePosixPath

if not AUTORIZAR_MONTAJE or not DRIVE_ROOT.is_dir():
    raise RuntimeError('Primero autoriza y monta Drive en la celda 2.')
zip_candidates = [DRIVE_ROOT / RUTA_ZIP_EN_DRIVE, DRIVE_ROOT / 'Vision-IA_Colab_A.zip']
PACKAGE_ZIP = next((p for p in zip_candidates if p.is_file()), None)
if PACKAGE_ZIP is None:
    raise FileNotFoundError('No se encontró el ZIP. Súbelo a Mi unidad/Vision-IA o corrige RUTA_ZIP_EN_DRIVE.')

EXTRACTION_ROOT = Path(tempfile.mkdtemp(prefix='vision_ia_', dir='/content'))
with zipfile.ZipFile(PACKAGE_ZIP) as bundle:
    for member in bundle.infolist():
        parts = PurePosixPath(member.filename).parts
        if not parts or parts[0] != 'Vision-IA' or '..' in parts or '\\' in member.filename or PurePosixPath(member.filename).is_absolute():
            raise ValueError('El ZIP contiene una ruta no admitida.')
        if ((member.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('El ZIP contiene un enlace simbólico no admitido.')
    bundle.extractall(EXTRACTION_ROOT)
PROJECT_DIR = EXTRACTION_ROOT / 'Vision-IA'
package_index = json.loads((PROJECT_DIR / 'PACKAGE_MANIFEST.json').read_text(encoding='utf-8'))
for entry in package_index['files']:
    path = (PROJECT_DIR / entry['path']).resolve()
    if not path.is_relative_to(PROJECT_DIR.resolve()):
        raise ValueError('Ruta fuera del paquete.')
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    if path.stat().st_size != entry['bytes'] or digest.hexdigest() != entry['sha256']:
        raise ValueError('Archivo incompleto o modificado: ' + entry['path'])
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Proyecto:', PROJECT_DIR)
print('Paquete verificado:', package_index['image_count'], 'imágenes, sin duplicar originales.')


## 4 · Instalar dependencias de entrenamiento

Se conserva el PyTorch con CUDA del entorno si satisface los requisitos. No instales aquí la variante CPU de Windows. No se instala Streamlit ni OpenCV y no se descarga ningún peso.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], cwd=PROJECT_DIR, check=True)
# El entrenamiento se lanzará en un proceso nuevo con las dependencias instaladas.


## 5 · Verificar las seis categorías y sus cantidades

Ejecuta esta celda. Deben aparecer **1.953 imágenes de entrenamiento y 421 de validación**, distribuidas entre seis categorías. Se comprueban los hashes de train/val. Esta celda no abre fotos test ni calcula métricas del modelo.


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.inspect_training_data', '--config', 'config/train_A.json', '--verify-hashes'], cwd=PROJECT_DIR, check=True)
DATA_READY = True


## 6 · Elegir parámetros y carpeta de resultados

Valores iniciales: hasta 30 épocas, batch 32 y learning rate 0.001. Se detiene antes si macro F1 de validación no mejora significativamente durante seis épocas. Puedes modificar los tres parámetros. Cada ejecución usa una carpeta nueva en Drive; el mejor modelo y el historial se guardan allí durante el entrenamiento.


In [ ]:
EPOCHS = 30 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}
from datetime import datetime, timezone
RUN_NAME = 'A_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RESULTS_DIR = DRIVE_ROOT / 'Vision-IA' / 'resultados' / RUN_NAME
print('Resultados futuros:', RESULTS_DIR)
print('Esta celda aún no entrena.')


## 7 · Entrenar — ejecutar esta celda inicia el entrenamiento real

Ejecuta esta celda cuando estés listo. Se crea una CNN aleatoria propia con 89.686 parámetros y se utilizan exclusivamente train/val. La GPU asignada se usa mediante CUDA. No se ha ejecutado esta celda al preparar el proyecto.

No reutilices el mismo directorio para otro entrenamiento. Para una ejecución nueva, vuelve a ejecutar la celda 6. Si se interrumpe la sesión, los resultados ya guardados permanecen en Drive; no se implementa reanudación del optimizador.


In [ ]:
if not globals().get('DATA_READY', False):
    raise RuntimeError('Primero verifica los datos en la celda 5.')
if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU CUDA. Selecciona GPU y vuelve a comprobar el entorno.')
command = [sys.executable, '-m', 'scripts.train', '--config', 'config/train_A.json',
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE),
           '--device', 'cuda', '--num-workers', '2', '--output-dir', str(RESULTS_DIR)]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


## 8 · Consultar métricas y gráficas reales

Solo funciona después de tu entrenamiento. La selección del checkpoint usa macro F1 de validación. No se presenta accuracy de prueba en este notebook.


In [ ]:
from IPython.display import display, Image as DisplayImage
summary_path = RESULTS_DIR / 'training_summary.json'
if not summary_path.is_file():
    raise FileNotFoundError('Primero ejecuta el entrenamiento; todavía no hay resultados finales.')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
history = json.loads((RESULTS_DIR / 'history.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Última época:', history[-1])
display(DisplayImage(filename=str(RESULTS_DIR / 'training_curves.png')))
print('Mejor modelo:', RESULTS_DIR / 'best_model.pth')


## 9 · Descargar los resultados para Windows

El checkpoint y los historiales ya están en Drive. Esta celda crea un ZIP de resultados y ofrece su descarga. Conserva su config.json y class_to_idx.json junto con best_model.pth.


In [ ]:
import shutil
from google.colab import files
if not (RESULTS_DIR / 'best_model.pth').is_file():
    raise FileNotFoundError('No hay un modelo entrenado para descargar.')
results_zip = shutil.make_archive(str(RESULTS_DIR.parent / RESULTS_DIR.name), 'zip', root_dir=RESULTS_DIR)
print('Copia en Drive:', results_zip)
files.download(results_zip)


## Evaluación final y limitaciones

Cuando hayas fijado el modelo y los hiperparámetros usando únicamente validación, ejecuta por separado `python -m scripts.evaluate --checkpoint RUTA/best_model.pth --device cpu`. En Windows usa el Python de tu entorno. El evaluador reserva las 417 imágenes test para ese momento y produce precision, recall y F1 por categoría, accuracy global y matriz de confusión.

En A, orgánico procede solo de RealWaste y las otras cinco categorías de TrashNet. Existe sesgo potencial por origen, fondo e iluminación. Las métricas internas no demuestran generalización a una cámara real. Revisar las candidatas visuales y evaluar posteriormente con otras fuentes. El experimento B queda fuera de este paquete. Consulta docs/DATASETS.md y docs/TRAINING.md.
